In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import joblib

# Project folders
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

# Existing verified artifacts
HISTORICAL_PATH = PROCESSED_DIR / "historical_matches.csv"
MODEL_FEATURES_PATH = PROCESSED_DIR / "model_features.csv"

MODEL_PATH = MODELS_DIR / "premier_league_model.joblib"
FEATURE_SCHEMA_PATH = MODELS_DIR / "feature_schema.json"
METADATA_PATH = MODELS_DIR / "model_metadata.json"

print("Setup complete.")

In [ ]:
files_to_check = {
    "Historical matches": HISTORICAL_PATH,
    "Model features": MODEL_FEATURES_PATH,
    "Production model": MODEL_PATH,
    "Feature schema": FEATURE_SCHEMA_PATH,
    "Model metadata": METADATA_PATH,
}

for name, path in files_to_check.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:20} {status:8} {path}")

In [ ]:
# Load production artifacts
model = joblib.load(MODEL_PATH)

with open(FEATURE_SCHEMA_PATH, "r") as f:
    feature_schema = json.load(f)

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

print("=== PRODUCTION MODEL ===")
print(f"Model type:       {type(model).__name__}")
print(f"Features:         {len(feature_schema)}")
print(f"Classes:          {model.classes_}")

print("\n=== SAVED METADATA ===")
print(f"Model name:       {metadata['model_name']}")
print(f"Test matches:     {metadata['test_matches']}")
print(f"Test period:      {metadata['test_start']} to {metadata['test_end']}")
print(f"Log loss:         {metadata['log_loss']}")
print(f"Market log loss:  {metadata['market_log_loss']}")

print("\n=== FEATURE SCHEMA ===")
for i, feature in enumerate(feature_schema, start=1):
    print(f"{i:02d}. {feature}")

In [ ]:
# Load existing historical match data
historical = pd.read_csv(HISTORICAL_PATH)

# Parse dates properly
historical["date"] = pd.to_datetime(historical["date"], utc=True)

print("=== HISTORICAL DATA ===")
print(f"Total matches: {len(historical):,}")
print(f"Date range:    {historical['date'].min()} to {historical['date'].max()}")

print("\n=== COMPETITIONS ===")
print(historical["competition"].value_counts())

print("\n=== PREMIER LEAGUE BY SEASON ===")
pl_historical = historical[
    historical["competition"] == "premier_league"
].copy()

print(pl_historical["season"].value_counts().sort_index())

# Current 2026/27 Premier League matches
current_season = (
    pl_historical[
        pl_historical["season"] == "2026_27"
    ]
    .sort_values("date")
    .copy()
)

print("\n=== 2026/27 PREMIER LEAGUE ===")
print(f"Matches currently stored: {len(current_season)}")

if len(current_season) > 0:
    print(f"First match: {current_season['date'].min()}")
    print(f"Last match:  {current_season['date'].max()}")

    print("\n=== MOST RECENT STORED MATCHES ===")
    print(
        current_season[
            [
                "date",
                "home_team",
                "away_team",
                "home_goals",
                "away_goals"
            ]
        ]
        .tail(10)
        .to_string(index=False)
    )

In [ ]:
# Raw match statistics required to keep the production features updated
required_raw_stats = [
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners",
]

print("=== REQUIRED RAW MATCH STATISTICS ===")

for col in required_raw_stats:
    exists = col in historical.columns

    if exists:
        missing = historical[col].isna().sum()
        print(f"{col:30} FOUND   missing: {missing:,}")
    else:
        print(f"{col:30} MISSING COLUMN")


print("\n=== 2026/27 COVERAGE ===")

for col in required_raw_stats:
    if col in current_season.columns:
        available = current_season[col].notna().sum()

        print(
            f"{col:30} "
            f"{available:>2}/{len(current_season)} available"
        )